In [ ]:
import os
import json
import gc
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm

# ==========================
# KONFIGURASI PATH
# ==========================
CSV_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv'
HDF5_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.hdf5'
ZHI_GENG_JSON = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Benchmark_ STEAD 3C_ test n15275 r100/STEAD data, test n15275 r100.json'

OUTPUT_JSON = '/Volumes/Extreme SSD/stream_stead/data_stead/STEAD_25000_MCUQuake_1C.json'
OUTPUT_META = '/Volumes/Extreme SSD/stream_stead/data_stead/STEAD_25000_metadata.csv'

NUM_EQ = 25000
NUM_NOISE = 25000
NUM_POINTS = 700
NORM_POINTS = 900

# ==========================
# FUNGSI UTILITAS
# ==========================

def safe_max_abs(x):
    x = np.asarray(x, dtype=float)
    m = np.max(np.abs(x)) if x.size > 0 else 0.0
    return m if m > 0 else 1e-8

# ==========================
# FASE 1: LOAD METADATA & ANTI-LEAKAGE
# ==========================

print("[INFO] Membaca metadata STEAD dari CSV...")
df_raw = pd.read_csv(CSV_PATH, low_memory=False)
df_filtered = df_raw[df_raw['trace_category'].isin(['earthquake_local', 'noise'])]

print("[INFO] Membaca daftar trace yang sudah dipakai Zhi Geng...")
with open(ZHI_GENG_JSON, 'r') as f:
    zhi_geng_data = json.load(f)

zhi_geng_traces = set(zhi_geng_data.keys()) if isinstance(zhi_geng_data, dict) else set(zhi_geng_data)
df_unseen = df_filtered[~df_filtered['trace_name'].isin(zhi_geng_traces)]

df_eq = df_unseen[df_unseen['trace_category'] == 'earthquake_local']
df_noise = df_unseen[df_unseen['trace_category'] == 'noise']

print(f"[INFO] Kandidat gempa   : {len(df_eq):,}")
print(f"[INFO] Kandidat noise   : {len(df_noise):,}")

df_eq_sample = df_eq.sample(n=min(NUM_EQ, len(df_eq)), random_state=42)
df_noise_sample = df_noise.sample(n=min(NUM_NOISE, len(df_noise)), random_state=42)

df_final = pd.concat([df_eq_sample, df_noise_sample]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"[INFO] Total target JSON: {len(df_final):,}")

trace_names = df_final['trace_name'].to_numpy()
trace_categories = df_final['trace_category'].to_numpy()
p_arrivals = df_final['p_arrival_sample'].fillna(-1).to_numpy().astype(np.int32)

del df_raw, df_filtered, df_unseen, df_eq, df_noise, df_eq_sample, df_noise_sample
gc.collect()

# ==========================
# FASE 2: BANGUN JSON + METADATA
# ==========================

dataset_json = {}
metadata_rows = []
n_skipped = 0

print("[INFO] Membaca waveform dari HDF5 dan membangun JSON 5000...")
with h5py.File(HDF5_PATH, 'r') as f_h5:
    data_group = f_h5['data']

    for idx in tqdm(range(len(trace_names)), desc="Building STEAD_5000 JSON"):
        trace_id = trace_names[idx]
        category = trace_categories[idx]
        p_arrival = p_arrivals[idx]

        if trace_id not in data_group:
            n_skipped += 1
            continue

        raw_wave = data_group[trace_id][:, 2].astype(np.float32)
        detrended = raw_wave - np.mean(raw_wave)

        if category == 'earthquake_local':
            if p_arrival < 0:
                n_skipped += 1
                continue

            start_idx = int(p_arrival)
            end_idx = start_idx + NUM_POINTS
            norm_end_idx = start_idx + NORM_POINTS

            if end_idx > len(detrended) or norm_end_idx > len(detrended):
                n_skipped += 1
                continue

            z_win = detrended[start_idx:end_idx]
            norm_win = detrended[start_idx:norm_end_idx]
            norm_val = safe_max_abs(norm_win)
            z_norm = z_win / norm_val

            noise_win = detrended[:NUM_POINTS]
            noise_norm_win = detrended[:NORM_POINTS]
            noise_norm_val = safe_max_abs(noise_norm_win)
            z_noise_norm = noise_win / noise_norm_val

            label_type = "se"

        else:
            if len(detrended) < max(NUM_POINTS, NORM_POINTS):
                n_skipped += 1
                continue

            z_win = detrended[:NUM_POINTS]
            norm_win = detrended[:NORM_POINTS]
            norm_val = safe_max_abs(norm_win)
            z_norm = z_win / norm_val

            if len(detrended) >= NORM_POINTS + NUM_POINTS:
                noise_start = NORM_POINTS
                noise_end = noise_start + NUM_POINTS
                noise_win = detrended[noise_start:noise_end]
                noise_norm_win = detrended[noise_start:noise_start + NORM_POINTS]
            else:
                noise_win = detrended[:NUM_POINTS]
                noise_norm_win = detrended[:NORM_POINTS]

            noise_norm_val = safe_max_abs(noise_norm_win)
            z_noise_norm = noise_win / noise_norm_val

            label_type = "no"

        dataset_json[trace_id] = {
            "Z": z_norm.tolist(),
            "Z_noise": z_noise_norm.tolist(),
            "type": label_type
        }

        metadata_rows.append({
            "id_json": trace_id,
            "trace_name": trace_id,
            "type": label_type,
            "trace_category": category,
            "p_arrival_sample": int(p_arrival),
            "sampling_rate": 100,
            "norm_val": float(norm_val),
            "z_max": float(np.max(np.abs(z_norm))),
            "z_noise_max": float(np.max(np.abs(z_noise_norm))),
            "window_start": int(start_idx if label_type=="se" else 0),
            "window_end": int(end_idx if label_type=="se" else NUM_POINTS),
            "noise_start": 0,
            "noise_end": NUM_POINTS,
            "source": "STEAD",
            "leakage_flag": 0
        })

print(f"[INFO] Total entry JSON: {len(dataset_json):,}")
print(f"[INFO] Total trace dilewati: {n_skipped:,}")

# ==========================
# FASE 3: SIMPAN JSON + CSV
# ==========================

os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)

with open(OUTPUT_JSON, 'w') as f_out:
    json.dump(dataset_json, f_out)

df_meta = pd.DataFrame(metadata_rows)
df_meta.to_csv(OUTPUT_META, index=False)

print(f"[SUKSES] JSON STEAD 5000 tersimpan di: {OUTPUT_JSON}")
print(f"[SUKSES] Metadata CSV tersimpan di: {OUTPUT_META}")


In [ ]:
import json

with open('/Volumes/Extreme SSD/stream_stead/data_stead/STEAD_MCQUAKE_5000_unseen.json') as f:
    data = json.load(f)

print(list(data.items())[:5])


In [ ]:
import json
import numpy as np

def inspect_uuss_structure(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)
        # Ambil satu sampel
        sample_key = list(data.keys())[0]
        sample = data[sample_key]
        
        print("=== ANALISIS STRUKTUR DATA UUSS ===")
        print(f"Keys yang tersedia: {list(sample.keys())}")
        print(f"Tipe label: {sample['type']}")
        
        # Cek dimensi array Z (apakah konsisten 700?)
        z_data = np.array(sample['Z'])
        print(f"Dimensi Z: {z_data.shape}")
        
        # Cek statistik dasar untuk normalisasi
        print(f"Mean Z: {np.mean(z_data):.4f}, Std Z: {np.std(z_data):.4f}")
        
        # Cek apakah ada Z_noise
        if 'Z_noise' in sample:
            print(f"Dimensi Z_noise: {np.array(sample['Z_noise']).shape}")
        else:
            print("Z_noise tidak ditemukan dalam sampel ini.")

# Jalankan inspeksi
UUSS_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Benchmark_ UUSS 3C_ test n2222 r100/UUSS 3C data, test n2222 r100.json"
inspect_uuss_structure(UUSS_PATH)

In [ ]:
import json
import numpy as np

def inspect_data_structure(filepath, label_name):
    with open(filepath, 'r') as f:
        data = json.load(f)
        
        # Ambil sampel pertama
        keys = list(data.keys())
        sample = data[keys[0]]
        
        print(f"=== ANALISIS {label_name} ===")
        print(f"Keys yang tersedia: {list(sample.keys())}")
        print(f"Tipe label (type): {sample.get('type')}")
        
        # Cek dimensi Z dan Z_noise
        z_data = np.array(sample['Z'])
        print(f"Dimensi Z: {z_data.shape}")
        
        if 'Z_noise' in sample:
            z_noise = np.array(sample['Z_noise'])
            print(f"Dimensi Z_noise: {z_noise.shape}")
        
        # Cek statistik untuk normalisasi
        print(f"Mean Z: {np.mean(z_data):.6f}")
        print(f"Std Z: {np.std(z_data):.6f}")
        print("-" * 30)

# Path File
UUSS_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Benchmark_ UUSS 3C_ test n2222 r100/UUSS 3C data, test n2222 r100.json"
STEAD_PATH = "/Volumes/Extreme SSD/stream_stead/data_stead/STEAD_25000_MCUQuake_1C.json"

inspect_data_structure(UUSS_PATH, "DATA UUSS")
inspect_data_structure(STEAD_PATH, "DATA STEAD")

In [ ]:
import os
import json
import gc
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm

# ==========================
# KONFIGURASI
# ==========================
CSV_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv'
HDF5_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.hdf5'
ZHI_GENG_JSON = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Benchmark_ UUSS 3C_ test n2222 r100/UUSS 3C data, test n2222 r100.json'

OUTPUT_JSON = '/Volumes/Extreme SSD/stream_stead/data_stead/STEAD_50000_Harmonized_Final.json'
OUTPUT_META = '/Volumes/Extreme SSD/stream_stead/data_stead/STEAD_50000_metadata_final.csv'

NUM_EQ = 10000
NUM_NOISE = 10000
TARGET_STD = 0.189108 
LEN_Z = 5000
LEN_NOISE = 1000

# ==========================
# FASE 1: LOAD METADATA
# ==========================
print("[INFO] Membaca metadata...")
df_raw = pd.read_csv(CSV_PATH, low_memory=False)
df_filtered = df_raw[df_raw['trace_category'].isin(['earthquake_local', 'noise'])]

with open(ZHI_GENG_JSON, 'r') as f:
    zhi_geng_traces = set(json.load(f).keys())

df_unseen = df_filtered[~df_filtered['trace_name'].isin(zhi_geng_traces)]
df_final = pd.concat([
    df_unseen[df_unseen['trace_category'] == 'earthquake_local'].sample(n=NUM_EQ, random_state=42),
    df_unseen[df_unseen['trace_category'] == 'noise'].sample(n=NUM_NOISE, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

# ==========================
# FASE 2: BUILD HARMONIZED DATASET
# ==========================
dataset_json = {}
metadata_rows = []

print("[INFO] Membangun dataset harmonized...")
with h5py.File(HDF5_PATH, 'r') as f_h5:
    data_group = f_h5['data']

    for idx, row in tqdm(df_final.iterrows(), total=len(df_final)):
        trace_id = row['trace_name']
        category = row['trace_category']
        
        if trace_id not in data_group: continue
        
        # Ambil waveform Z (channel 2)
        raw_wave = data_group[trace_id][:, 2].astype(np.float32)
        detrended = raw_wave - np.mean(raw_wave)
        
        # Potong atau Padding ke 5000
        z_win = detrended[:LEN_Z] if len(detrended) >= LEN_Z else np.pad(detrended, (0, LEN_Z - len(detrended)), 'constant')
        
        # Normalisasi Skala (Matching Std Dev UUSS)
        current_std = np.std(z_win) if np.std(z_win) > 0 else 1.0
        z_norm = (z_win / current_std) * TARGET_STD
        
        # Sintesis 3C Dummy (Replikasi Z ke N dan E)
        n_norm = z_norm.copy()
        e_norm = z_norm.copy()
        
        # Noise (1000 poin)
        noise_z = z_norm[:LEN_NOISE]
        noise_n = n_norm[:LEN_NOISE]
        noise_e = e_norm[:LEN_NOISE]
        
        label_type = "se" if category == 'earthquake_local' else "no"
        
        dataset_json[trace_id] = {
            "type": label_type,
            "Z": z_norm.tolist(),
            "N": n_norm.tolist(),
            "E": e_norm.tolist(),
            "Z_noise": noise_z.tolist(),
            "N_noise": noise_n.tolist(),
            "E_noise": noise_e.tolist(),
            "norm": float(np.max(np.abs(z_norm)))
        }

# ==========================
# FASE 3: SIMPAN
# ==========================
with open(OUTPUT_JSON, 'w') as f_out:
    json.dump(dataset_json, f_out)

print(f"[SUKSES] Dataset harmonized tersimpan: {OUTPUT_JSON}")

[INFO] Membaca metadata...
[INFO] Memproses batch 1...


100%|██████████| 5000/5000 [01:02<00:00, 80.48it/s]


[INFO] Memproses batch 2...


 47%|████▋     | 2326/5000 [00:27<00:32, 83.12it/s]


KeyboardInterrupt: 